# Notebook 3 — Personas & Simulation des Migrations

> Profilage des clusters en personas marketing, split temporel et simulation de la détection de migrations.

**Projet** : BDD #7 Sephora × Albert School | Groupe 5 | Case 2  
**Seed** : 42 (reproductibilité garantie)


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join(os.getcwd(), '..'))  # accès au package src/

RANDOM_SEED = 42

---

## 3.1 Profilage des clusters

Calcul des KPIs par cluster et comparaison vs moyenne globale.


### KPIs par cluster

CA moyen, panier, fréquence, recency, discount rate, top marques, top axes.


In [ ]:

import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils import DATA_PATH, MODELS_PATH
from src.clustering import load_model, preprocess
from src.personas import (profile_cluster, compute_delta_vs_global,
                           generate_persona_card, plot_radar_chart,
                           PERSONA_NAMES)

# Chargement modèle + features train (avec cluster)
model, scaler = load_model()
feat_train = pd.read_csv(os.path.join(DATA_PATH, "customer_features_train.csv"),
                          index_col="anonymized_card_code")

# Si colonne cluster absente, la recalculer
if "cluster" not in feat_train.columns:
    X, _, _ = preprocess(feat_train, scaler=scaler, fit=False)
    feat_train["cluster"] = model.predict(X)
    feat_train.to_csv(os.path.join(DATA_PATH, "customer_features_train.csv"))
    print("Clusters recalculés et sauvegardés")

# Profil des clusters
profile = profile_cluster(feat_train)
print(f"Clusters identifiés : {len(profile)}")
print("\n=== KPIs MOYENS PAR CLUSTER ===")
print(profile[["n_clients", "pct_clients", "monetary", "avg_basket",
               "frequency", "recency_days", "discount_ratio"]].round(2).to_string())


### Delta vs moyenne globale

Obligatoire : chaque KPI comparé à la moyenne du dataset.


In [ ]:

# Calcul des deltas vs moyenne globale
profile_with_delta = compute_delta_vs_global(profile)

print("=== DELTA vs MOYENNE GLOBALE (colonnes delta_*) ===")
delta_cols = [c for c in profile_with_delta.columns if c.startswith("delta_")]
print(profile_with_delta[delta_cols[:8]].round(1).to_string())

# Heatmap des deltas normalisés
kpi_base = ["monetary", "avg_basket", "frequency", "recency_days",
            "discount_ratio", "pct_estore", "icb_score"]
kpi_base = [c for c in kpi_base if c in profile.columns]

delta_vals = profile_with_delta[[f"delta_{c}_pct" for c in kpi_base
                                  if f"delta_{c}_pct" in profile_with_delta.columns]]

fig, ax = plt.subplots(figsize=(11, max(4, len(profile) * 0.7)))
im = ax.imshow(delta_vals.values, cmap="RdPu", aspect="auto", vmin=-100, vmax=100)
ax.set_xticks(range(len(delta_vals.columns)))
ax.set_yticks(range(len(delta_vals.index)))
ax.set_xticklabels([c.replace("delta_", "").replace("_pct", "") for c in delta_vals.columns],
                   rotation=30, ha="right")
ax.set_yticklabels([f"Cluster {c}" for c in delta_vals.index])

for i in range(len(delta_vals.index)):
    for j in range(len(delta_vals.columns)):
        val = delta_vals.iloc[i, j]
        sign = "+" if val >= 0 else ""
        color = "white" if abs(val) > 60 else "#1A1A1A"
        ax.text(j, i, f"{sign}{val:.0f}%", ha="center", va="center",
                fontsize=9, color=color, fontweight="bold")

plt.colorbar(im, ax=ax, label="Delta vs moyenne (%)")
ax.set_title("Delta KPIs vs Moyenne Globale par Cluster", fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/3_delta_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


### Comparaison avec RFM Sephora

Overlap entre nos clusters et les RFM_Segment_ID existants.


In [ ]:

from src.personas import compare_with_sephora_rfm

if "RFM_Segment_ID" in feat_train.columns:
    cross = compare_with_sephora_rfm(feat_train)
    print("=== OVERLAP CLUSTERS × RFM_SEGMENT_ID SEPHORA ===")
    print("(% des clients d'un cluster appartenant à chaque RFM_Segment_ID)")
    print(cross.round(1).to_string())

    # Insight
    print("\n→ Nos clusters ML recoupent et enrichissent les segments RFM Sephora")
    print("  → Présenter comme 'affinement granulaire' de la segmentation existante")
else:
    print("RFM_Segment_ID absent des features — impossible de faire la comparaison")


---

## 3.2 Création des fiches personas

Nommage et description narrative de chaque cluster.


### Nommage des personas

Nom évocateur pour le marketing + description en une phrase.


In [ ]:

# Fiches personas complètes
global_stats = profile.select_dtypes(include=[np.number]).mean()

persona_cards = []
print("=== FICHES PERSONAS ===\n")
for cl in sorted(profile.index):
    row = profile.loc[cl]
    card = generate_persona_card(cl, row, global_stats, feat_train)
    persona_cards.append(card)

    print(f"{'='*60}")
    print(f"CLUSTER {cl} — {card['name']}")
    print(f"  📊 {card['n_clients']:,} clients ({card['pct_clients']:.1f}%)")
    print(f"  💰 CA moyen       : {card['kpis']['monetary_mean']:.0f} € "
          f"({card['deltas_vs_global']['monetary']})")
    print(f"  🛒 Panier moyen   : {card['kpis']['avg_basket']:.0f} € "
          f"({card['deltas_vs_global']['avg_basket']})")
    print(f"  📅 Fréquence      : {card['kpis']['frequency']:.1f} tickets "
          f"({card['deltas_vs_global']['frequency']})")
    print(f"  ⏱️  Recency        : {card['kpis']['recency_days']:.0f} jours")
    print(f"  🎯 ICB Score      : {card['kpis']['icb_score']:.0f}/100")
    print(f"  💎 CLV estimée    : {card['kpis']['clv_estimated']:.0f} €")
    print(f"\n  RECOMMANDATIONS MARKETING :")
    for r in card["recommendations"]:
        print(f"    {r}")
    print()


### Radar charts

Visualisation 8 axes : Budget, Fidélité, Diversité, Premium, Digital, Promo, Skincare, Fragrance.


In [ ]:

# Radar charts pour tous les clusters
global_max = profile.select_dtypes(include=[np.number]).max().to_dict()

print("Génération des radar charts...")
radar_paths = []
for cl in sorted(profile.index):
    path = plot_radar_chart(profile.loc[cl], cl, global_max,
                            global_mean_row=global_stats)
    radar_paths.append(path)
    print(f"  Cluster {cl} ({PERSONA_NAMES.get(cl, '?')}) → {path}")

# Afficher tous les radars dans une grille
n_clusters = len(profile)
n_cols = min(3, n_clusters)
n_rows = (n_clusters + n_cols - 1) // n_cols

fig, axes_r = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 7 * n_rows),
                            subplot_kw={"polar": True})
if n_clusters == 1:
    axes_r = [[axes_r]]
elif n_rows == 1:
    axes_r = [axes_r]

axes_flat = [ax for row in axes_r for ax in row]

# Masquer les axes en trop
for ax in axes_flat[n_clusters:]:
    ax.set_visible(False)

fig.suptitle("Radar Charts — Tous les Personas Sephora", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
plt.savefig("../outputs/figures/3_radar_all_personas.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\n✓ Radar charts sauvegardés dans outputs/figures/")


---

## 3.3 Indice de Curiosité Beauté (ICB)

Score propriétaire 0–100 mesurant la propension à explorer.


### Calcul de l'ICB

Combinaison pondérée de brand_entropy, axe_diversity, new_brand_rate, channel_diversity.


In [ ]:

from src.personas import compute_beauty_curiosity_index

# ICB par cluster
icb = compute_beauty_curiosity_index(feat_train)
feat_train["icb_score_computed"] = icb

icb_by_cluster = feat_train.groupby("cluster")["icb_score_computed"].mean()
print("=== ICB MOYEN PAR CLUSTER ===")
for cl in sorted(icb_by_cluster.index):
    name = PERSONA_NAMES.get(cl, f"Cluster {cl}")
    score = icb_by_cluster[cl]
    print(f"  Cluster {cl} ({name:35s}) : ICB = {score:5.1f}/100")

# Clients à fort ICB (> 70)
n_high_icb = (feat_train["icb_score_computed"] >= 70).sum()
print(f"\nClients ICB >= 70 : {n_high_icb:,} ({n_high_icb/len(feat_train)*100:.1f}%)")
print("→ Ces clients sont les 'Beauty Addicts' à fort potentiel de CLV")


### Corrélation ICB / CLV

Montrer que les clients à fort ICB ont un CA supérieur à la moyenne.


In [ ]:

# Corrélation ICB / CA
fig, ax = plt.subplots(figsize=(9, 5))

# Scatter ICB vs monetary (sample pour lisibilité)
sample = feat_train.sample(min(5000, len(feat_train)), random_state=42)
scatter = ax.scatter(sample["icb_score_computed"], sample["monetary"],
                     alpha=0.3, s=15, c=sample["cluster"],
                     cmap="RdPu", vmin=0, vmax=model.n_clusters - 1)

# Ligne de régression
from numpy.polynomial.polynomial import polyfit
mask_valid = ~(sample["icb_score_computed"].isna() | sample["monetary"].isna())
x_vals = sample.loc[mask_valid, "icb_score_computed"].values
y_vals = sample.loc[mask_valid, "monetary"].values.clip(0, np.percentile(y_vals if len(x_vals) > 0 else [0], 99) if len(x_vals) > 0 else 0)
if len(x_vals) > 10:
    coefs = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, np.polyval(coefs, x_line), color="#FF0066", lw=2, label="Tendance")

corr = feat_train["icb_score_computed"].corr(feat_train["monetary"])
ax.set_title(f"ICB Score vs CA total — corrélation r = {corr:.3f}", fontweight="bold")
ax.set_xlabel("ICB Score (0–100)")
ax.set_ylabel("CA total client (€)")
plt.colorbar(scatter, ax=ax, label="Cluster")
ax.legend()
fig.tight_layout()
plt.savefig("../outputs/figures/3_icb_vs_ca.png", dpi=150, bbox_inches="tight")
plt.show()

# Groupes ICB
icb_bins = pd.cut(feat_train["icb_score_computed"], bins=[0, 30, 60, 70, 100],
                   labels=["ICB 0-30", "ICB 30-60", "ICB 60-70", "ICB 70-100"])
print("CA moyen par tranche ICB :")
print(feat_train.groupby(icb_bins)["monetary"].mean().round(0))


---

## 3.4 Split temporel et simulation

Division Jan–Juin / Juil–Sep, replay des transactions.


### Initialisation du feature store

Construire {client_id: profil} à partir de features_train.csv.


In [ ]:

from src.migration_detector import build_feature_store as build_fs

# Feature store initialisé depuis les features train
feature_store = build_fs(os.path.join(DATA_PATH, "customer_features_train.csv"))

# Clusters initiaux
initial_clusters = {str(cid): int(lbl)
                    for cid, lbl in feat_train["cluster"].items()}

print(f"Feature store : {len(feature_store):,} clients")
print(f"Initial clusters : {len(initial_clusters):,} clients")
print(f"Distribution initiale : {pd.Series(initial_clusters).value_counts().sort_index().to_dict()}")


### Replay Juil–Sep

Traiter les transactions chronologiquement, mettre à jour les profils.


In [ ]:

from src.migration_detector import replay_transactions

# Replay Juil–Sep chronologique
txn_test_path = os.path.join(DATA_PATH, "transactions_test.csv")
print(f"Fichier transactions test : {txn_test_path}")
print(f"Existe : {os.path.exists(txn_test_path)}")

migrations, final_clusters = replay_transactions(
    txn_test_path,
    feature_store,
    initial_clusters,
    model,
    scaler,
    verbose=True
)

print(f"\n✓ Replay terminé :")
print(f"  Migrations détectées : {len(migrations):,}")
print(f"  Clients trackés final : {len(final_clusters):,}")


### Détection des migrations

Comparer ancien et nouveau cluster pour chaque transaction.


In [ ]:

# Analyse des migrations détectées
if migrations:
    mig_df = pd.DataFrame(migrations)
    mig_df["date"] = pd.to_datetime(mig_df["date"])

    print("=== APERÇU MIGRATIONS ===")
    print(mig_df.head(10).to_string(index=False))

    print(f"\n=== STATISTIQUES ===")
    print(f"Total migrations     : {len(mig_df):,}")
    print(f"Clients migrés       : {mig_df['client_id'].nunique():,}")
    print(f"\nDirection :")
    print(mig_df["direction"].value_counts())
    print(f"\nFlux les plus fréquents (top 10) :")
    flux = (mig_df.groupby(["from_cluster", "to_cluster"])
                  .size().sort_values(ascending=False).head(10))
    print(flux)
else:
    print("Aucune migration détectée — vérifier le fichier transactions_test.csv")
    mig_df = pd.DataFrame(columns=["client_id", "txn_id", "date",
                                    "from_cluster", "to_cluster", "direction"])


---

## 3.5 Analyse des migrations

Matrice de transition et signaux précurseurs.


### Matrice de migration

Tableau NxN des flux entre clusters sur la période Juil–Sep.


In [ ]:

from src.migration_detector import build_migration_matrix, save_migrations, plot_migration_heatmap

# Matrice de migration
matrix = build_migration_matrix(migrations, model.n_clusters)
print("=== MATRICE DE MIGRATION (Juil–Sep 2025) ===")
print("(lignes = cluster d'origine | colonnes = cluster de destination)")
print(matrix.to_string())

# Taux de migration par cluster
if len(mig_df) > 0:
    for cl in sorted(range(model.n_clusters)):
        n_init = sum(1 for v in initial_clusters.values() if v == cl)
        n_migrated = len(mig_df[mig_df["from_cluster"] == cl]["client_id"].unique())
        if n_init > 0:
            print(f"  Cluster {cl} : {n_migrated:,} / {n_init:,} clients ont migré "
                  f"({n_migrated/n_init*100:.1f}%)")

# Sauvegarder
paths = save_migrations(migrations, matrix)
plot_migration_heatmap(matrix)
print(f"\n✓ Migrations sauvegardées → {paths}")


### Sankey diagram

Visualisation des flux de migration — figure principale de la présentation.


In [ ]:

from src.migration_detector import plot_sankey_migrations

# Sankey diagram (interactif HTML si plotly disponible)
sankey_path = plot_sankey_migrations(migrations, model.n_clusters)
print(f"✓ Sankey diagram → {sankey_path}")


### Signaux précurseurs

Quel type d'achat déclenche une migration ? Analyse par axe et marque.


In [ ]:

# Signaux précurseurs de migration
# Quelle est la dernière transaction avant chaque migration ?
if len(mig_df) > 0 and os.path.exists(txn_test_path):
    txn_test = pd.read_csv(txn_test_path,
                            dtype={"anonymized_card_code": str},
                            low_memory=False)
    txn_test["transactionDate"] = pd.to_datetime(
        txn_test["transactionDate"], format="%m/%d/%Y", errors="coerce")

    # Pour chaque migration, regarder les 1-2 achats précédents
    sample_migrations = mig_df.head(1000)  # échantillon pour l'analyse
    migrant_ids = sample_migrations["client_id"].unique()

    trigger_txns = txn_test[txn_test["anonymized_card_code"].isin(migrant_ids)]

    print("=== AXES D'ACHAT DÉCLENCHEURS DE MIGRATION ===")
    print("(transactions des clients ayant migré pendant la période test)")
    if "Axe_Desc" in trigger_txns.columns:
        axe_trigger = trigger_txns["Axe_Desc"].value_counts(normalize=True) * 100
        print(axe_trigger.round(1))

    print("\n=== MARQUES DÉCLENCHEURS (top 15) ===")
    if "brand" in trigger_txns.columns:
        brand_trigger = trigger_txns["brand"].value_counts().head(15)
        print(brand_trigger)

    print("\n→ Insight : si un client dormant achète [AXE X], probabilité de migration vers Cluster Y = ...%")
    print("  → Signal utilisable pour déclencher une séquence CRM proactive")
else:
    print("Pas de migrations détectées ou fichier test absent")

# Sauvegarder personas
from src.personas import save_personas
save_personas(profile_with_delta, persona_cards)
print("\n✓ Personas sauvegardées")
